# A3 corregido: Red híbrida CNN-LSTM (correcciones i, iii y v)

Cambios respecto a `A3_Red Híbrida CNN-LSTM.ipynb` (el original queda intacto):
1. La etiqueta es `was_preterm` (antes `was_early_preterm`), igual que en A1 y A2.
2. La prueba ya no es una sola partición estratificada por centro (`project`):
   se usan los pliegues comunes por paciente (`pliegues_por_paciente.csv`,
   5 pliegues x 5 repeticiones, estratificados por la etiqueta).
3. Se guardan las probabilidades por paciente de cada pliegue en
   `resultados/pred_A3.csv` para armar la Tabla 4.1.

Punto iii (entrenamiento y umbral):

4. La validación (25 % de las pacientes de entrenamiento) se separa **estratificada por la etiqueta**.
5. El punto de control (`ModelCheckpoint`) guarda los pesos con menor `val_loss`
   (antes `val_accuracy`), y se agrega `EarlyStopping` sobre `val_loss` (paciencia 50 épocas,
   máximo 500).
6. Se agrega ponderación de clases, w_c = N / (2·N_c), calculada con el entrenamiento de cada pliegue.
7. El umbral de decisión es el índice de Youden, max(TPR − FPR), calculado con las predicciones
   del conjunto de **validación**. Se congela y se aplica una sola vez al pliegue de prueba.
8. Al final: matriz de confusión por paciente (repetición 0) y la figura de selección del umbral
   sobre validación, para el Capítulo 4.

9. Bloque denso: se agregan las tres capas `Dense(100)` que describe la Sec. 3.5.3, cada una
   seguida de LeakyReLU (pendiente 0.01), BatchNormalization y Dropout(0.15). El original tenía
   `LeakyReLU(100)` sin capas densas, lo que ponía la pendiente de los negativos en 100.

10. `pad_sequences` usa `dtype='float32'` (antes el valor por omisión, `int32`, truncaba a 0 las
    similitudes `*_sim` y el `score` de los estados comunitarios).

Punto v (variantes de entrada, mismos pliegues, semillas, entrenamiento y umbral):

- `completa`: la secuencia de cada paciente (últimas 9 muestras, en orden cronológico). Es el modelo de la Tabla 4.1.
- `barajada`: las mismas muestras, con su orden permutado al azar dentro de cada paciente (semilla fija por paciente).
- `ultima_muestra`: solo la última muestra de cada paciente.
- `horizonte28`: solo las muestras con `collect_wk` ≤ 28 y sin la columna `collect_wk`. Las pacientes sin
  muestras hasta la semana 28 quedan fuera de esa variante.

Cada variante entrena 25 redes, así que con las 4 son 100 entrenamientos. Para correr solo el modelo
completo, deja `VARIANTES = ['completa']` en la celda de configuración.

La Conv1D y la LSTM **no** cambian.

`cst_valencia.csv` no está en el repositorio. Sus 16 columnas (CST, subCST, score y las
13 similitudes `*_sim`) están dentro de `cst_valencia_w_taxons.csv`, así que se toman de
ahí. Con eso se reproduce la tabla original: 1895 x 10158 y 10138 predictores.

**Google Colab:** sube la carpeta `CorreccionesTesis` (CSV, `comun.py`, `pliegues_por_paciente.csv`
y este notebook) a *Mi unidad* de Google Drive, activa la GPU (Entorno de ejecución → Cambiar
tipo de entorno → GPU) y corre todas las celdas. La primera celda monta Drive y entra a la carpeta.
En tu computadora esa celda no hace nada.

In [ ]:
# Solo en Google Colab: monta Google Drive y entra a la carpeta con los datos y comun.py.
# En una computadora local esta celda no hace nada.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/CorreccionesTesis'   # cámbiala si la subiste a otro lugar
    os.chdir(CARPETA)
    print('Trabajando en', os.getcwd())
except ImportError:
    print('No es Colab: se usa la carpeta actual,', os.getcwd())

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os, sys
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import random
import numpy as np
import pandas as pd
import tensorflow as tf
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.preprocessing import sequence
from tensorflow.keras import backend as K
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, confusion_matrix
import seaborn as sns

sys.path.insert(0, os.path.abspath('.'))
from comun import (leer_datos, RESULTADOS, ETIQUETA, cargar_pliegues, iterar_pliegues,
                   umbral_youden, metricas)
RESULTADOS.mkdir(exist_ok=True)
EPOCHS = int(os.environ.get('A3_EPOCHS', 500))
PACIENCIA = 50          # EarlyStopping: épocas sin mejorar val_loss antes de parar
# Variantes de entrada (punto v). 'completa' es el modelo de la Tabla 4.1.
VARIANTES = ['completa', 'barajada', 'ultima_muestra', 'horizonte28']
SEMANA_HORIZONTE = 28
print('Keras', keras.__version__, '| épocas:', EPOCHS)

In [ ]:
community = leer_datos('cst_valencia_w_taxons')
cols_cst = ['specimen', 'CST', 'subCST', 'score'] + [c for c in community.columns if c.endswith('_sim')]
alpha_diversity = community[cols_cst].copy()   # equivale a cst_valencia.csv del original
metadata = leer_datos('metadata')
phylotypes = leer_datos('phylotype_nreads.1e_1')

metadata.loc[metadata.age=='Unknown','age']='99'
CatVal = ["Below_18","Above_38","99","29-38","18_to_28"]
metadata_num = metadata.loc[~metadata["age"].isin(CatVal)]
metadata_num = metadata_num.astype({"age": float})
metadata_num.age = metadata_num.age.round()
metadata_num = metadata_num.astype({"age": int})
bins = pd.IntervalIndex.from_tuples([(0, 18), (18, 28), (28, 38),(38,80)])
metadata_num["age"] = pd.cut(metadata_num["age"], bins)
metadata_num['age'] = metadata_num['age'].astype(str)
metadata_num.loc[metadata_num.age=='(0, 18]','age']="Below_18"
metadata_num.loc[metadata_num.age=='(18, 28]','age']="18_to_28"
metadata_num.loc[metadata_num.age=='(28, 38]','age']="29-38"
metadata_num.loc[metadata_num.age=='(38, 80]','age']="Above_38"
metadata_car = metadata.loc[metadata["age"].isin(CatVal)]
metadata = pd.concat([metadata_num, metadata_car], axis=0, ignore_index=True)

combineddata = alpha_diversity.merge(community, left_on='specimen', right_on='specimen')
combineddata = combineddata.merge(metadata, left_on='specimen', right_on='specimen')
combineddata = combineddata.merge(phylotypes, left_on='specimen', right_on='specimen')
X = combineddata
print(X.shape)

In [ ]:
Cols1 = X.columns[1:422]
Cols2 = ["collect_wk"]
Cols3 = X.columns[435:len(X.columns)]
ColstoGet = np.append(Cols1,Cols2)
ColstoGet = np.append(ColstoGet,Cols3)
for c in ['index_y', 'CST', 'subCST', 'NIH Ethnicity Category', 'NIH Racial Category', 'race',
          # Variables de desenlace: nunca deben ser predictores (son booleanas y el filtro
          # numérico ya las quitaba en el original; se excluyen de forma explícita).
          'delivery_wk', 'was_term', 'was_preterm', 'was_early_preterm']:
    ColstoGet = ColstoGet[ColstoGet != c]

Predictors = np.append(ColstoGet, 'participant_id')
Predictors = np.append(Predictors, [ETIQUETA, 'project'])   # CAMBIO: was_preterm
print(Predictors.shape)

In [ ]:
X_ids = combineddata["specimen"]
X = X.loc[:, Predictors]
X["specimen_person"] = X_ids.str.split('-').str[0]
X = X.loc[:, X.dtypes.apply(lambda dt: np.issubdtype(dt, np.number)) | (X.columns == "specimen_person")]
for col in [ETIQUETA, 'participant_id']:                   # CAMBIO: was_preterm
    if col not in X.columns:
        X[col] = combineddata[col]

persons = X["specimen_person"].unique()
secuencias, lstmDat_y = [], []        # una tabla por paciente (sus muestras en orden de colecta)
for person in persons:
    loopDat = X.loc[X["specimen_person"] == person]
    loopy = loopDat[ETIQUETA].unique()                     # CAMBIO: was_preterm
    loopDat = loopDat.drop(['specimen_person', ETIQUETA, 'participant_id'], axis=1)
    loopDat = loopDat.sort_values('collect_wk', kind='stable')   # orden cronológico explícito
    secuencias.append(loopDat)
    lstmDat_y.append(loopy)

lstmDat_y_flat = [item for sublist in lstmDat_y for item in sublist]
lstmDat_y_flat = np.where(np.array(lstmDat_y_flat) == True, 1, 0)

max_sequence_length = 9
columnas = secuencias[0].columns
n_features = len(columnas)
# Datos de muestreo por paciente (para pred_A3.csv; no son entradas del modelo)
n_muestras = np.array([len(s) for s in secuencias])
ultima_collect_wk = np.array([s['collect_wk'].iloc[-1] for s in secuencias])
print("pacientes:", len(persons), "| variables:", n_features,
      "| pretérmino:", lstmDat_y_flat.sum(), "de", len(lstmDat_y_flat))
assert n_features == 10138

## Variantes de entrada (punto v)
`construir_entrada(variante)` arma el tensor (pacientes × 9 × variables) de cada variante. En todas, las
secuencias se completan con ceros al inicio (*padding*) hasta 9 posiciones, como en el modelo original.

In [ ]:
def construir_entrada(variante):
    """Devuelve (X, incluidas, n_entrada): tensor de la variante, máscara de pacientes que entran
    y número de muestras que recibe la red de cada paciente."""
    lista, incluidas, n_entrada = [], [], []
    for i, s in enumerate(secuencias):
        s = s.iloc[-max_sequence_length:]                    # últimas 9 muestras, como el original
        if variante == 'barajada':
            orden = np.random.default_rng(SEED + i).permutation(len(s))
            s = s.iloc[orden]
        elif variante == 'ultima_muestra':
            s = s.iloc[-1:]
        elif variante == 'horizonte28':
            s = secuencias[i]
            s = s[s['collect_wk'] <= SEMANA_HORIZONTE].iloc[-max_sequence_length:]
            s = s.drop(columns='collect_wk')
        elif variante != 'completa':
            raise ValueError(variante)
        incluidas.append(len(s) > 0)
        n_entrada.append(len(s))
        lista.append(s.to_numpy(dtype='float32') if len(s) else
                     np.zeros((0, n_features - (variante == 'horizonte28')), dtype='float32'))
    Xv = sequence.pad_sequences(lista, maxlen=max_sequence_length, dtype='float32')
    return Xv, np.array(incluidas), np.array(n_entrada)

for v in VARIANTES:
    Xv, inc, _ = construir_entrada(v)
    print(f'{v:15s} tensor {Xv.shape} | pacientes incluidas {inc.sum()} de {len(inc)}')
    del Xv

## Pliegues comunes por paciente
Antes: `train_test_split(..., stratify=project)`, una sola partición. Ahora: los mismos
25 pliegues por paciente que usan la regresión logística y el Random Forest.

In [ ]:
pliegues = cargar_pliegues()
indice = {p: i for i, p in enumerate(persons)}
assert set(indice) == set(pliegues.participant_id)

def construir_modelo(n_feat=n_features):
    K.clear_session()
    x = layers.Input(shape=(max_sequence_length, n_feat))
    model = layers.Conv1D(filters=16, kernel_size=3, padding='same', activation='elu')(x)
    model = layers.MaxPooling1D(pool_size=2)(model)
    model = layers.LSTM(100)(model)
    for _ in range(3):
        # Capa densa de 100 neuronas + LeakyReLU con pendiente 0.01 (punto iii; Sec. 3.5.3).
        # El original usaba LeakyReLU(100) sin capa densa: eso fijaba la pendiente en 100.
        model = layers.Dense(100)(model)
        model = layers.LeakyReLU(0.01)(model)
        model = layers.BatchNormalization()(model)
        model = layers.Dropout(0.15)(model)
    model = layers.Dense(1, activation='sigmoid')(model)
    myModel = keras.Model(x, model)
    myModel.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return myModel

construir_modelo().summary()

In [ ]:
predicciones, pred_val, filas, historias, excluidas = [], [], [], {}, {}
for variante in VARIANTES:
    X_var, incluidas, n_entrada = construir_entrada(variante)
    excluidas[variante] = int((~incluidas).sum())
    print(f'\n=== Variante: {variante} | pacientes excluidas: {excluidas[variante]} ===', flush=True)
    nombre_modelo = 'CNN-LSTM' if variante == 'completa' else f'CNN-LSTM ({variante})'
    for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
        tr = np.array(sorted(indice[p] for p in ids_tr if incluidas[indice[p]]))
        te = np.array(sorted(indice[p] for p in ids_te if incluidas[indice[p]]))
        X_train, X_test = X_var[tr], X_var[te]
        y_train, y_test = lstmDat_y_flat[tr], lstmDat_y_flat[te]
        # Validación: 25 % de las pacientes de entrenamiento, estratificada por la etiqueta (punto iii)
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                          stratify=y_train, random_state=SEED)
        # Ponderación de clases w_c = N / (2 N_c), con el entrenamiento del pliegue
        n, n1 = len(y_train), int(y_train.sum())
        pesos = {0: n / (2 * (n - n1)), 1: n / (2 * n1)}

        keras.utils.set_random_seed(SEED)
        myModel = construir_modelo(X_var.shape[2])
        checkpoint_filepath = f'./best_model_{variante}_r{rep}_f{fold}.weights.h5'
        cb = [tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, save_weights_only=True,
                                                 monitor='val_loss', mode='min', save_best_only=True),
              tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=PACIENCIA)]
        h = myModel.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_val, y_val),
                        batch_size=32, callbacks=cb, class_weight=pesos, verbose=0)
        historias[(variante, rep, fold)] = h.history
        myModel.load_weights(checkpoint_filepath)
        os.remove(checkpoint_filepath)

        # Umbral de Youden con la VALIDACIÓN; luego se aplica una sola vez a prueba (punto iii)
        yval_pred = myModel.predict(X_val, verbose=0).ravel()
        umbral = umbral_youden(y_val, yval_pred)
        ypred = myModel.predict(X_test, verbose=0).ravel()
        m = metricas(y_test, ypred, umbral)
        mejor_epoca = int(np.argmin(h.history['val_loss'])) + 1
        filas.append(dict(modelo=nombre_modelo, variante=variante, rep=rep, fold=fold, **m,
                          epocas_corridas=len(h.history['loss']), mejor_epoca=mejor_epoca))
        pred_val.append(pd.DataFrame({'variante': variante, 'rep': rep, 'fold': fold,
                                      'y': y_val, 'prob': yval_pred}))
        predicciones.append(pd.DataFrame({
            'participant_id': persons[te], 'fold': fold, 'rep': rep, 'variante': variante,
            'y_true': y_test, 'y_score': ypred,
            'n_muestras': n_muestras[te], 'ultima_collect_wk': ultima_collect_wk[te],
            'n_muestras_entrada': n_entrada[te]}))
        print(f"{variante} rep {rep} fold {fold}: AUC-ROC {m['auc_roc']:.3f}  AUC-PR {m['auc_pr']:.3f}  "
              f"recall {m['recall']:.2f}  umbral {umbral:.3f}  épocas {len(h.history['loss'])} "
              f"(mejor {mejor_epoca})", flush=True)
    del X_var
    # Se guarda al terminar cada variante, por si la corrida se interrumpe
    pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A3.csv', index=False)
    pd.concat(pred_val).to_csv(RESULTADOS / 'pred_val_A3.csv', index=False)
    pd.DataFrame(filas).to_csv(RESULTADOS / 'metricas_A3.csv', index=False)

res = pd.DataFrame(filas)
print('Pacientes excluidas por variante:', excluidas)
res.groupby('variante', sort=False)[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1',
                                      'umbral']].agg(['mean', 'std']).round(3)

In [ ]:
# Curvas de pérdida del primer pliegue (antes se dibujaban en vivo con PlotLosses)
h = historias[('completa', 0, 0)]
plt.plot(h['loss'], label='loss'); plt.plot(h['val_loss'], label='v_loss')
plt.xlabel('Épocas'); plt.ylabel('Pérdida (Loss)'); plt.title('Evolución de la pérdida (completa, rep 0, pliegue 0)')
plt.legend(); plt.savefig(RESULTADOS / 'LossEpocas_A3.png', dpi=200); plt.show()

## Matriz de confusión por paciente (modelo completo, repetición 0)
En la repetición 0 cada una de las 616 pacientes cae en prueba exactamente una vez. Cada paciente se
clasifica con el umbral de su pliegue, calculado con la validación de ese pliegue.

In [ ]:
# Solo el modelo completo (el de la Tabla 4.1)
pred = pd.concat(predicciones)
pred = pred[pred.variante == 'completa']
umbrales = res[res.variante == 'completa'].set_index(['rep', 'fold']).umbral
r0 = pred[pred.rep == 0].copy()
assert r0.participant_id.nunique() == len(r0) == 616
r0['pred'] = (r0.y_score >= r0.apply(lambda f: umbrales[(f.rep, f.fold)], axis=1)).astype(int)
cm = confusion_matrix(r0.y_true, r0.pred, labels=[0, 1])
pd.DataFrame(cm, index=['Real: término', 'Real: pretérmino'],
             columns=['Predicho: término', 'Predicho: pretérmino']).to_csv(RESULTADOS / 'matriz_confusion_A3.csv')
tn, fp, fn, tp = cm.ravel()
print(f'VN={tn}  FP={fp}  FN={fn}  VP={tp}  | recall={tp/(tp+fn):.3f}  especificidad={tn/(tn+fp):.3f}  '
      f'precisión={tp/(tp+fp):.3f}')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Término', 'Pretérmino'], yticklabels=['Término', 'Pretérmino'])
plt.title('Matriz de confusión por paciente (CNN-LSTM)'); plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.savefig(RESULTADOS / 'MatrixConfusion_A3_paciente.png', dpi=200); plt.show()

## Selección del umbral sobre validación (repetición 0, pliegue 0)
Sensibilidad (TPR) y especificidad (1 − FPR) en función del umbral, calculadas con el conjunto de
validación. La línea vertical marca el umbral de Youden, el que maximiza TPR − FPR.

In [ ]:
v = pd.concat(pred_val)
v = v[(v.variante == 'completa') & (v.rep == 0) & (v.fold == 0)]
fpr, tpr, thr = roc_curve(v.y, v.prob)
thr = np.minimum(thr, 1.0)
u = umbrales[(0, 0)]
plt.figure(figsize=(7, 5))
plt.plot(thr, tpr, label='Sensibilidad (TPR)')
plt.plot(thr, 1 - fpr, color='red', label='Especificidad (1 - FPR)')
plt.axvline(u, color='k', ls='--', label=f'Umbral de Youden = {u:.3f}')
plt.xlabel('Umbral de decisión'); plt.ylabel('Tasa'); plt.title('Selección del umbral (validación, completa, rep 0, pliegue 0)')
plt.legend(); plt.tight_layout(); plt.savefig(RESULTADOS / 'Umbral_validacion_A3.png', dpi=200); plt.show()
print(res.groupby('variante', sort=False)[['umbral', 'epocas_corridas', 'mejor_epoca']].mean().round(3))

## Punto v: incertidumbre, controles y ablaciones

Esta sección no entrena nada: solo lee los archivos de `resultados/` (`pred_A3.csv`, `pred_A2.csv` y
`pred_A2_media.csv`), así que puede correrse sola después de A2 y de las celdas anteriores.

1. **Intervalos de confianza del AUC.** Para cada repetición se juntan las predicciones fuera de pliegue
   (cada paciente aparece una vez) y se calcula el AUC con su IC 95 % por *bootstrap* estratificado por
   pacientes (2000 remuestreos) y por Hanley–McNeil.
2. **Comparaciones pareadas pliegue a pliegue** del modelo completo contra cada variante, el Random Forest
   y el Random Forest de la media por paciente: diferencia media de AUC, IC 95 % con la prueba *t*
   corregida de Nadeau–Bengio (los pliegues comparten pacientes de entrenamiento, así que la *t*
   ordinaria subestima la varianza) y cuántos pliegues gana cada modelo.
3. **Control de tiempo de muestreo.** Regresión logística con solo dos variables por paciente: número de
   muestras y semana de la última colecta, en los mismos pliegues. Si la red no lo supera, su AUC no puede
   atribuirse al microbioma.
4. **Subcohorte con al menos 3 muestras**, donde existe una trayectoria que aprender.

Criterio fijado antes de ver los resultados: se afirma que el orden temporal aporta información solo si la
red completa supera a la variante barajada **y** al Random Forest de la media, con un IC 95 % de la
diferencia que excluya 0, **y** además supera al control de tiempo de muestreo.

In [ ]:
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

N_BOOT = 2000
p3 = pd.read_csv(RESULTADOS / 'pred_A3.csv').rename(columns={'y_true': 'y', 'y_score': 'prob'})
p3['modelo'] = np.where(p3.variante == 'completa', 'CNN-LSTM', 'CNN-LSTM (' + p3.variante + ')')
p2 = pd.read_csv(RESULTADOS / 'pred_A2.csv').assign(modelo='Random Forest')
p2m = pd.read_csv(RESULTADOS / 'pred_A2_media.csv').assign(modelo='Random Forest (media por paciente)')

# Datos de muestreo por paciente (iguales en todas las variantes)
muestreo = (p3[p3.variante == 'completa'].drop_duplicates('participant_id')
            .set_index('participant_id')[['n_muestras', 'ultima_collect_wk']])

# Control de tiempo de muestreo: regresión logística con n_muestras y ultima_collect_wk
pacientes_ctrl = cargar_pliegues().drop_duplicates('participant_id').set_index('participant_id')
Xc = muestreo.loc[pacientes_ctrl.index].values
yc = pacientes_ctrl[ETIQUETA].astype(int).values
idc = pacientes_ctrl.index.values
pc = []
for rep, fold, ids_tr, ids_te in iterar_pliegues(cargar_pliegues()):
    tr, te = np.isin(idc, list(ids_tr)), np.isin(idc, list(ids_te))
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc[tr], yc[tr])
    pc.append(pd.DataFrame({'participant_id': idc[te], 'y': yc[te], 'prob': lr.predict_proba(Xc[te])[:, 1],
                            'rep': rep, 'fold': fold}))
pctrl = pd.concat(pc).assign(modelo='Control: n.º de muestras y última semana')

todas = pd.concat([p3[['participant_id', 'y', 'prob', 'rep', 'fold', 'modelo']], p2, p2m, pctrl],
                  ignore_index=True)
todas = todas.merge(muestreo, left_on='participant_id', right_index=True, how='left')
orden_modelos = list(dict.fromkeys(todas.modelo))
print(todas.groupby('modelo', sort=False).participant_id.nunique())

### 1. AUC con intervalo de confianza (predicciones fuera de pliegue de cada repetición)

In [ ]:
def hanley_mcneil(auc, n1, n0):
    q1, q2 = auc / (2 - auc), 2 * auc**2 / (1 + auc)
    se = np.sqrt((auc * (1 - auc) + (n1 - 1) * (q1 - auc**2) + (n0 - 1) * (q2 - auc**2)) / (n1 * n0))
    return auc - 1.96 * se, auc + 1.96 * se

def auc_bootstrap(y, prob, semilla):
    rng = np.random.default_rng(semilla)
    i1, i0 = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    n1, n0 = len(i1), len(i0)
    b = []
    for _ in range(N_BOOT):
        # AUC de Mann-Whitney con rangos (igual a roc_auc_score, pero mucho más rápido)
        r = stats.rankdata(np.r_[prob[rng.choice(i1, n1)], prob[rng.choice(i0, n0)]])
        b.append((r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
    return np.percentile(b, [2.5, 97.5])

filas_ic = []
for (modelo, rep), d in todas.groupby(['modelo', 'rep'], sort=False):
    y, prob = d.y.values, d.prob.values
    auc = roc_auc_score(y, prob)
    b_lo, b_hi = auc_bootstrap(y, prob, semilla=rep)
    h_lo, h_hi = hanley_mcneil(auc, y.sum(), (1 - y).sum())
    filas_ic.append(dict(modelo=modelo, rep=rep, n=len(y), positivas=int(y.sum()), auc=auc,
                         ic_boot_inf=b_lo, ic_boot_sup=b_hi, ic_hm_inf=h_lo, ic_hm_sup=h_hi))
ic = pd.DataFrame(filas_ic)
ic.to_csv(RESULTADOS / 'v_auc_ic_por_repeticion.csv', index=False)

# Resumen por modelo: media de las 5 repeticiones y AUC por pliegue (25 pliegues)
por_pliegue = (todas.groupby(['modelo', 'rep', 'fold'], sort=False)
               .apply(lambda d: roc_auc_score(d.y, d.prob)).rename('auc').reset_index())
resumen = ic.groupby('modelo', sort=False)[['auc', 'ic_boot_inf', 'ic_boot_sup', 'ic_hm_inf', 'ic_hm_sup']].mean()
resumen['auc_pliegues_media'] = por_pliegue.groupby('modelo', sort=False).auc.mean()
resumen['auc_pliegues_de'] = por_pliegue.groupby('modelo', sort=False).auc.std()
resumen['pliegues_auc_>0.5'] = por_pliegue.groupby('modelo', sort=False).auc.apply(lambda a: int((a > 0.5).sum()))
resumen['pliegues_auc_>0.7'] = por_pliegue.groupby('modelo', sort=False).auc.apply(lambda a: int((a > 0.7).sum()))
resumen.round(3).to_csv(RESULTADOS / 'v_resumen_auc.csv')
resumen.round(3)

### 2. Comparaciones pareadas contra la CNN-LSTM completa (25 pliegues)

In [ ]:
def comparar(a, b, datos=por_pliegue, k=5, r=5):
    # Diferencia AUC(a) - AUC(b) por pliegue; IC con la t corregida de Nadeau-Bengio
    pa = datos[datos.modelo == a].set_index(['rep', 'fold']).auc
    pb = datos[datos.modelo == b].set_index(['rep', 'fold']).auc
    d = (pa - pb).dropna()
    n = len(d)
    var_corr = (1 / n + 1 / (k - 1)) * d.var(ddof=1)     # n_prueba / n_entrenamiento = 1/(k-1)
    t = d.mean() / np.sqrt(var_corr)
    mitad = stats.t.ppf(0.975, n - 1) * np.sqrt(var_corr)
    return dict(comparacion=f'{a}  -  {b}', pliegues=n, dif_media=d.mean(), ic_inf=d.mean() - mitad,
                ic_sup=d.mean() + mitad, p_valor=2 * stats.t.sf(abs(t), n - 1),
                gana_a=int((d > 0).sum()), gana_b=int((d < 0).sum()))

completa = 'CNN-LSTM'
comparaciones = pd.DataFrame([comparar(completa, m) for m in orden_modelos if m != completa])
comparaciones.round(3).to_csv(RESULTADOS / 'v_comparaciones_pareadas.csv', index=False)
comparaciones.round(3)

### 3. Subcohorte con al menos 3 muestras por paciente

In [ ]:
sub = todas[todas.n_muestras >= 3]
print('Pacientes con >= 3 muestras:', sub.participant_id.nunique(), '| pretérmino:',
      sub.drop_duplicates('participant_id').y.sum())
por_pliegue_sub = (sub.groupby(['modelo', 'rep', 'fold'], sort=False)
                   .apply(lambda d: roc_auc_score(d.y, d.prob) if d.y.nunique() == 2 else np.nan)
                   .rename('auc').reset_index().dropna())
resumen_sub = por_pliegue_sub.groupby('modelo', sort=False).auc.agg(['mean', 'std', 'count'])
comparaciones_sub = pd.DataFrame([comparar(completa, m, por_pliegue_sub)
                                  for m in orden_modelos if m != completa])
resumen_sub.round(3).to_csv(RESULTADOS / 'v_resumen_auc_3muestras.csv')
comparaciones_sub.round(3).to_csv(RESULTADOS / 'v_comparaciones_pareadas_3muestras.csv', index=False)
display(resumen_sub.round(3)); comparaciones_sub.round(3)

### 4. Figuras para el Capítulo 4 y lectura del criterio

In [ ]:
# AUC por pliegue de cada modelo
fig, ax = plt.subplots(figsize=(9, 5))
datos = [por_pliegue[por_pliegue.modelo == m].auc.values for m in orden_modelos]
ax.boxplot(datos, vert=False, showfliers=False)
for i, a in enumerate(datos, start=1):
    ax.scatter(a, np.random.default_rng(i).normal(i, 0.06, len(a)), s=12, alpha=0.6)
ax.set_yticks(range(1, len(orden_modelos) + 1), orden_modelos)
ax.axvline(0.5, color='gray', ls='--', lw=1); ax.axvline(0.7, color='gray', ls=':', lw=1)
ax.set_xlabel('AUC-ROC por pliegue (5 x 5, por paciente)'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(RESULTADOS / 'AUC_pliegues_v.png', dpi=200); plt.show()

# Curva ROC fuera de pliegue de la CNN-LSTM completa (repetición 0), con su IC
d0 = todas[(todas.modelo == completa) & (todas.rep == 0)]
fpr, tpr, _ = roc_curve(d0.y, d0.prob)
f = ic[(ic.modelo == completa) & (ic.rep == 0)].iloc[0]
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'CNN-LSTM: AUC = {f.auc:.2f} (IC 95 %: {f.ic_boot_inf:.2f}-{f.ic_boot_sup:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('Tasa de falsos positivos (FPR)'); plt.ylabel('Tasa de verdaderos positivos (TPR)')
plt.title('Curva ROC por paciente (predicciones fuera de pliegue, rep 0)')
plt.legend(loc='lower right'); plt.tight_layout(); plt.savefig(RESULTADOS / 'CurvaROC_A3_paciente.png', dpi=200); plt.show()

# Criterio fijado de antemano
def supera(m, tabla=comparaciones):
    f = tabla[tabla.comparacion == f'{completa}  -  {m}'].iloc[0]
    return f.ic_inf > 0
criterio = {m: supera(m) for m in ['CNN-LSTM (barajada)', 'Random Forest (media por paciente)',
                                     'Control: n.º de muestras y última semana']}
print('¿La CNN-LSTM completa supera, con IC de la diferencia > 0, a...')
for m, ok in criterio.items():
    print(f'   {m}: {"sí" if ok else "no"}')
print('=> El orden temporal aporta información demostrable:', 'SÍ' if all(criterio.values()) else 'NO')